# Pilot: MEANtools functional clusters (tomato) vs PlantMetWiki

**Question.** MEANtools links metabolites and gene transcripts that co-vary across samples and groups them into *functional clusters*.
How many of these gene–metabolite associations can be mapped onto PlantMetWiki? And when they can, do the gene and the metabolite meet in the same PlantCyc pathway?

**Why.** The goal is an evidence model for statements like *"gene G is associated with metabolite M because they are in the same MEANtools functional cluster"*.
A link to PlantMetWiki would then either support the association (a known pathway) or flag it as new.
This is evidence case 2 (*paired omics M-T correlations*) in the main [README](../../README.md).

**Data.** Tomato leaves infected with fungal pathogens (Jeon et al. 2020), from the MEANtools case study:
- MEANtools output: `Jeon_fungal.sqlite`, Zenodo [15697913](https://zenodo.org/records/15697913)
- metabolomics: MetaboLights MTBLS1039, positive ion mode
- RNA-seq: BioProject PRJNA509154

The 2.6 GB bacterial-treatment output is supported by the script but not used here.

**Approach.** Both sides of every gene–metabolite pair in a cluster are mapped to PlantMetWiki pathways:

| Side | Tier | Rule |
|---|---|---|
| gene | `gene_locus` | the Solyc locus is itself a gene of the pathway (PlantMetWiki stores ~20 SGN loci) |
| gene | `gene_uniprot` | the gene's UniProt protein is an enzyme of the pathway |
| gene | `gene_ec` | the gene's EC number (UniProt) catalyses a reaction of the pathway |
| metabolite | `mass_match` | the feature m/z matches a pathway metabolite as a common positive-mode adduct (±10 ppm). MS1 only, so weak evidence. |

All logic is in [`match_meantools.py`](match_meantools.py). It reuses the PlantMetWiki snapshots and pathway classes of the [first pilot](../metabolomics_overlap/).
Numbers in the text are computed in the cells.

## How to reproduce

```bash
cd pilots/meantools_overlap
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python -r requirements.txt   # pinned versions
```

Open this notebook with the `.venv` kernel and **Run All**, or run
`.venv/bin/jupyter nbconvert --to notebook --execute --inplace explore_results.ipynb`.

| Input | How it is obtained | Frozen? |
|---|---|---|
| MEANtools output `Jeon_fungal.sqlite` | downloaded from Zenodo, MD5 checked | yes (record v3.0.0) |
| XCMS feature table (exact m/z) | downloaded from the Sattely-lab GitHub at a pinned commit, MD5 checked | yes |
| Tomato UniProt entries (locus, accession, EC) | UniProt REST, saved as `data/uniprot_tomato.tsv.gz` | yes: snapshot 2026-09-29 |
| PlantMetWiki: pathway ECs, UniProt ids, Solyc loci, metabolite InChIs | SPARQL, saved as `data/plantmetwiki_*.csv` | yes: snapshot 2026-09-29 |
| PlantMetWiki metabolites, pathways, classes | from the first pilot (`../metabolomics_overlap/data/`) | yes |

## 1. Setup and provenance

In [1]:
import platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import match_meantools as mm

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 200)

def say(text):
    display(Markdown(text))

DATASET = "fungal"
DECAY_RATE = 10     # see section 2 for why
res = mm.build(DATASET, DECAY_RATE)
pairs, gmap, mmap, candidates = res["pairs"], res["gmap"], res["mmap"], res["candidates"]
clusters, uniprot, pmw_genes = res["clusters"], res["uniprot"], res["pmw_genes"]

In [2]:
display(pd.Series({"python": platform.python_version(), "pandas": pd.__version__,
                   "matplotlib": matplotlib.__version__}, name="version").to_frame())
mm.snapshot_info(DATASET)

,version
python,3.12.9
pandas,3.0.6
matplotlib,3.11.2


,size_MB,md5,last_modified
file,,,
meantools_overlap/data/plantmetwiki_metabolite_inchi.csv,0.78,1809f1eb53c8fced27ef4318bb7580cf,2026-09-29 15:38
meantools_overlap/data/plantmetwiki_pathway_ec.csv,0.16,c04f8fb61fae1997cc92200066b88abe,2026-09-29 15:37
meantools_overlap/data/plantmetwiki_pathway_solyc.csv,0.00,7d970b3a2478c64a9502bac432de901f,2026-09-29 15:37
meantools_overlap/data/plantmetwiki_pathway_uniprot.csv,0.17,2b3bf89ba635f66bdea7ba9311e53b1c,2026-09-29 15:37
meantools_overlap/data/plantmetwiki_pc_pathways.csv,0.12,44c9227c88e23113545d905d62d1f245,2026-09-29 15:37
meantools_overlap/data/uniprot_tomato.tsv.gz,0.51,1114426184a0276d70c60692a53cc1cb,2026-09-29 15:36
meantools_overlap/data/xcms_feature_table.csv,1.25,39ba472fe131d7b56874fa65a8c6c9f6,2026-09-29 15:39
meantools_overlap/data/Jeon_fungal.sqlite,28.17,b5080825a727b99c660a4a5437ec596d,2026-09-29 15:24
metabolomics_overlap/data/plantmetwiki_metabolite_pathways.csv,1.42,fd6c2f38d8c5f5231227e46b80b55e59,2026-09-29 14:51


## 2. The MEANtools output

- **Correlations.** MEANtools computes Pearson correlations between each metabolite feature and each gene.
- **Mutual ranks.** It turns the mutual rank (MR) of each pair into an edge weight, exp(−(MR−1)/DR). The decay rate DR (5, 10, 25, 50) controls how quickly weights fall off.
- **Clusters.** ClusterONE then finds functional clusters on the weighted network and gives each cluster a p-value.

In [3]:
dr_summary = mm.cluster_summary(DATASET)
dr_summary

,clusters,clusters_P<0.05,median_size,genes,metabolites
decay_rate,,,,,
5,1372,9,5.0,1657,134
10,2391,111,5.0,3406,137
25,2368,7,5.0,4631,135
50,2478,4,6.0,4674,137


In [4]:
corr = res["corr"]
best = dr_summary["clusters_P<0.05"].idxmax()
say(f"""**Findings**
- The correlation table links **{corr['metabolite'].nunique()}** metabolite features to **{corr['gene'].nunique()}** genes
  ({len(corr)} pairs, Pearson r between {corr['correlation'].min():.2f} and {corr['correlation'].max():.2f}).
- Only a few clusters are statistically significant (ClusterONE p < 0.05). Decay rate **{best}** has by far the most
  (**{dr_summary.loc[best, 'clusters_P<0.05']}** of {dr_summary.loc[best, 'clusters']}), so we use DR = {DECAY_RATE} below.
  Change `DECAY_RATE` in the setup cell to compare.""")

**Findings**
- The correlation table links **138** metabolite features to **5380** genes
  (121043 pairs, Pearson r between -0.91 and 0.99).
- Only a few clusters are statistically significant (ClusterONE p < 0.05). Decay rate **10** has by far the most
  (**111** of 2391), so we use DR = 10 below.
  Change `DECAY_RATE` in the setup cell to compare.

**Interpretation**
- The metabolite side is small: only ~140 features were retained, the ones induced by the fungal treatments. The gene side covers thousands of genes, so every cluster is dominated by genes.
- **The falcarindiol features are missing.** Jeon et al. mark two XCMS features as falcarindiol, M283T2030 and M225T2033. Neither is in the fungal correlation output, so the flagship pathway of the study can't be tested with this file (see section 7).

## 3. Mapping genes to PlantMetWiki

In [5]:
genes = pd.Series(sorted(set(pairs["locus"])))
in_up = genes.isin(uniprot["locus"])
with_ec = genes.isin(uniprot.dropna(subset=["ec"])["locus"])
gene_cov = pd.DataFrame({
    "genes": [len(genes), in_up.sum(), with_ec.sum()] + [gmap.loc[gmap["tier"] == t, "locus"].nunique() for t in mm.GENE_TIERS] + [gmap["locus"].nunique()],
}, index=["genes in clusters (DR %d)" % DECAY_RATE, "…with a UniProt entry", "…with a complete EC number"]
        + [f"mapped via {t}" for t in mm.GENE_TIERS] + ["mapped to ≥ 1 PlantMetWiki pathway (any tier)"])
gene_cov["%"] = (100 * gene_cov["genes"] / len(genes)).round(1)
gene_cov

,genes,%
genes in clusters (DR 10),3406,100.0
…with a UniProt entry,3033,89.0
…with a complete EC number,654,19.2
mapped via gene_locus,14,0.4
mapped via gene_uniprot,1,0.0
mapped via gene_ec,298,8.7
mapped to ≥ 1 PlantMetWiki pathway (any tier),311,9.1


In [6]:
# The genes that PlantMetWiki lists by their Solyc locus
gmap[gmap["tier"] == "gene_locus"].drop_duplicates(["locus", "plantcyc_id"])

,locus,tier,plantcyc_id,pathway,via
0,Solyc01g073640,gene_locus,PWY18C3-1,"cholesterol biosynthesis (plants, early side-chain reductase)",SDR
1,Solyc01g073640,gene_locus,PWY-2541,phytosterol biosynthesis (plants),SDR
2,Solyc01g091320,gene_locus,PWY18C3-1,"cholesterol biosynthesis (plants, early side-chain reductase)",SMO3
3,Solyc02g081730,gene_locus,PWY18C3-1,"cholesterol biosynthesis (plants, early side-chain reductase)",3βHSD-2
4,Solyc02g081730,gene_locus,PWY-2541,phytosterol biosynthesis (plants),3βHSD-2
5,Solyc02g086180,gene_locus,PWY66-341,cholesterol biosynthesis I,C5-SD2
6,Solyc02g086180,gene_locus,PWY18C3-1,"cholesterol biosynthesis (plants, early side-chain reductase)",C5-SD2
7,Solyc06g005750,gene_locus,PWY18C3-1,"cholesterol biosynthesis (plants, early side-chain reductase)",SMO4
8,Solyc06g082980,gene_locus,PWY18C3-1,"cholesterol biosynthesis (plants, early side-chain reductase)","8,7 SI"
9,Solyc06g082980,gene_locus,PWY-2541,phytosterol biosynthesis (plants),"8,7 SI"


**Comments**
- **Bridging IDs.** MEANtools uses ITAG2 gene models (e.g. `Solyc04g025520.1`), while UniProt cross-references newer models (`…3.1`). We match on the unversioned locus. A few loci were retired between genome versions and have no UniProt entry.
- **The EC route is the main bridge, but it's coarse.** Only about 1 in 5 cluster genes has a complete EC number. An EC number says what *kind* of reaction a gene may catalyse, not that this tomato gene acts in that pathway. Broad ECs such as dehydrogenases and transferases map to many pathways.
- **The direct `gene_locus` tier is the strongest but tiny.** PlantMetWiki lists only ~20 tomato loci, mostly GAME genes (α-tomatine biosynthesis) and acylsugar ASAT genes.
- **The UniProt tier contributes almost nothing.** PlantCyc's curated enzymes are mostly from Arabidopsis and other species, not tomato.

## 4. Mapping metabolite features to PlantMetWiki

### 4a. Why we don't use MEANtools' own annotation table

The SQLite file contains `NPDB_annotations`, which is meant to be mass-matched candidate structures per feature. Its rows are not mass-consistent.

In [7]:
npdb = res["npdb"].copy()
npdb["Ion_name"] = npdb["Ion_name"].str.strip()
mh = npdb[npdb["Ion_name"] == "M+H"]
consistent = (mh["mz"] - (mh["monoisotopic_mass"] + 1.007276)).abs() < 0.01
say(f"""- Every feature has exactly one row per adduct type ({npdb['Ion_name'].nunique()} adducts × {npdb['ms_name'].nunique()} features).
- For the [M+H]+ rows, the feature m/z equals the structure's mass + 1.0073 in only **{consistent.sum()} of {len(mh)}** rows.
- The same structure is assigned to features with very different m/z, even for the same adduct.""")
npdb.loc[npdb["inchi"].str.startswith("InChI=1S/C9H15N5O4"), ["ms_name", "mz", "Ion_name", "monoisotopic_mass"]].head(6)

- Every feature has exactly one row per adduct type (41 adducts × 149 features).
- For the [M+H]+ rows, the feature m/z equals the structure's mass + 1.0073 in only **4 of 149** rows.
- The same structure is assigned to features with very different m/z, even for the same adduct.

,ms_name,mz,Ion_name,monoisotopic_mass
182,M201T449,201.005915,M+3Na,257.112404
183,M201T449,201.005915,M+IsoProp+H,257.112404
184,M201T449,201.005915,M+ACN+Na,257.112404
185,M201T449,201.005915,M+Hac-H,257.112404
186,M202T625,202.177504,M+3Na,257.112404
187,M202T625,202.177504,M+IsoProp+H,257.112404


**Comment.** In the MEANtools code (`queryMassNPDB.py`), the lower tolerance bound is computed with the *upper* function (`mm_low = mm.transform(checkP)`).
That may be related to the problem. Either way, this table can't be used as structural evidence. **Worth reporting to the MEANtools authors.**

### 4b. Our own MS1 mass matching against PlantMetWiki

- **Metabolite masses.** For every PlantMetWiki metabolite, we compute the neutral monoisotopic mass from its InChI formula layer. That layer describes the neutral parent even for charged entries.
- **Matching.** Each feature's exact m/z (from the XCMS table) is compared with [M+H]+, [M+Na]+, [M+NH4]+, [M+K]+ and [M+H−H2O]+ at ±10 ppm.

In [8]:
feats = res["features"]
n_match = candidates["feature"].nunique()
per_feat = candidates.groupby("feature")["key14"].nunique()
say(f"""**Findings**
- **{n_match}** of {len(feats)} clustered features (**{round(100 * n_match / len(feats))}%**) match at least one PlantMetWiki metabolite by mass;
  **{mmap['feature'].nunique()}** of these match a metabolite that belongs to a pathway.
- Candidates per matched feature: median **{per_feat.median():.0f}**, max **{per_feat.max()}**.""")
display(candidates["adduct"].value_counts().to_frame("matches per adduct"))
candidates.sort_values("feature")[["feature", "mz", "adduct", "ppm_error", "pmw_names"]].head(30)

**Findings**
- **47** of 137 clustered features (**34%**) match at least one PlantMetWiki metabolite by mass;
  **43** of these match a metabolite that belongs to a pathway.
- Candidates per matched feature: median **1**, max **8**.

,matches per adduct
adduct,
[M+NH4]+,36
[M+Na]+,19
[M+H]+,18
[M+H-H2O]+,11
[M+K]+,5


,feature,mz,adduct,ppm_error,pmw_names
0,M147T671,147.043079,[M+H]+,-6.64,coumarin
18,M147T671,147.043079,[M+Na]+,9.73,salicyl alcohol
19,M147T671,147.043079,[M+Na]+,9.73,orcinol
78,M147T671,147.043079,[M+H-H2O]+,-6.64,3-phenyl-2-oxopropanoate
79,M147T671,147.043079,[M+H-H2O]+,-6.64,4-coumarate
20,M147T671,147.043079,[M+Na]+,9.73,guaiacol
81,M147T671,147.043079,[M+H-H2O]+,-6.64,cis-2-coumarate | trans-2-coumarate
80,M147T671,147.043079,[M+H-H2O]+,-6.64,caffealdehyde
82,M180T321,180.100982,[M+H-H2O]+,-5.12,methyl ecgonone
38,M180T321,180.100982,[M+NH4]+,-5.11,4-hydroxybenzalacetone


**Interpretation**
- A mass match only says that a feature *could be* that metabolite. Isomers have identical masses, and at ±10 ppm unrelated formulas can match by chance, especially with five adducts.
- Retention time and MS/MS spectra would be needed for a real identification (as in the first pilot). The tomato data in MetaboLights include MS/MS, so running SIRIUS or a spectral library search is an obvious next step.

## 5. Do gene and metabolite of a cluster pair meet in a PlantMetWiki pathway?

- **Pairs.** For every gene × metabolite pair within a functional cluster, we check whether the gene's pathways (any tier) and the metabolite's pathways overlap.
- **Background.** We ask the same question for *all* combinations of mapped genes × mapped metabolites. This gives the fraction you'd expect by chance, and it is deterministic, so no random seed is needed.

In [9]:
order = ["both mapped, shared pathway", "both mapped, no shared pathway", "gene only", "metabolite only", "neither mapped"]
mapping = pairs["mapping"].value_counts().reindex(order).fillna(0).astype(int).to_frame("pairs")
mapping["%"] = (100 * mapping["pairs"] / len(pairs)).round(1)
sig = pairs[pairs["cluster_P"] < 0.05]
mapping["pairs in significant clusters"] = sig["mapping"].value_counts().reindex(order).fillna(0).astype(int)
mapping

,pairs,%,pairs in significant clusters
mapping,,,
"both mapped, shared pathway",0,0.0,0
"both mapped, no shared pathway",721,3.3,82
gene only,1725,7.9,158
metabolite only,6842,31.4,1189
neither mapped,12529,57.4,2639


In [10]:
n_bg_shared, n_bg = mm.expected_shared_fraction(pairs, gmap, mmap)
both = pairs["mapping"].str.startswith("both")
obs = (pairs["mapping"] == "both mapped, shared pathway").sum()
exp_frac = n_bg_shared / n_bg if n_bg else 0
say(f"""**Findings (DR = {DECAY_RATE})**
- Of **{len(pairs)}** gene–metabolite pairs in functional clusters, **{both.sum()}** ({100 * both.mean():.1f}%) have both partners mapped to PlantMetWiki.
- **{obs}** of them share a PlantMetWiki pathway.
- By chance (all combinations of mapped genes × mapped metabolites) we would expect {100 * exp_frac:.2f}%, i.e. **≈{exp_frac * both.sum():.1f}** pairs.""")

**Findings (DR = 10)**
- Of **21817** gene–metabolite pairs in functional clusters, **721** (3.3%) have both partners mapped to PlantMetWiki.
- **0** of them share a PlantMetWiki pathway.
- By chance (all combinations of mapped genes × mapped metabolites) we would expect 0.73%, i.e. **≈5.3** pairs.

In [11]:
# The same question for every decay rate
rows = []
for dr in mm.DECAY_RATES:
    r = mm.build(DATASET, dr)
    p = r["pairs"]
    nb, nt = mm.expected_shared_fraction(p, r["gmap"], r["mmap"])
    b = p["mapping"].str.startswith("both").sum()
    rows.append({"decay_rate": dr, "pairs": len(p), "both mapped": b,
                 "shared pathway (observed)": int((p["mapping"] == "both mapped, shared pathway").sum()),
                 "shared pathway (expected by chance)": round(b * nb / nt, 1) if nt else 0})
per_dr = pd.DataFrame(rows).set_index("decay_rate")
per_dr

,pairs,both mapped,shared pathway (observed),shared pathway (expected by chance)
decay_rate,,,,
5,10629,356,1,3.0
10,21817,721,0,5.3
25,40447,1314,4,8.9
50,45738,1805,7,12.7


**Interpretation.** At the pathway level, the fungal MEANtools clusters **do not recover PlantMetWiki pathways** any better than chance.
Three bottlenecks explain most of this. They are not necessarily a failure of MEANtools:
1. **Metabolite identity.** Features are matched by mass only, and many fungus-induced metabolites (oxylipins, falcarindiol and other acetylenic fatty acids) have no PlantCyc pathway at all.
2. **Gene function.** Only ~20% of genes have an EC number, and EC-based pathway membership is broad.
3. **Scope.** PlantMetWiki covers curated, largely primary and well-known specialized pathways. MEANtools is designed to find *new* pathways, like the falcarindiol cluster, that by definition aren't in PlantMetWiki yet.

So for an evidence model, a MEANtools association should stand on its **own** evidence: correlation, mutual rank and cluster significance.
PlantMetWiki support is an optional, extra piece of evidence, and its absence doesn't make the association wrong.

## 6. Gene-side coherence: do genes of one cluster share PlantMetWiki pathways?

Even without metabolite identities, we can test whether MEANtools clusters group genes of the same known pathway.
We count clusters with ≥ 2 genes mapped to the same PlantMetWiki pathway, and compare with 100 random reassignments of genes to clusters.
The shuffle keeps the cluster sizes and uses a fixed seed, so it is reproducible.

In [12]:
def coherent_clusters(cluster_genes, gene_pw, count="locus"):
    # clusters with >= 2 distinct `count` values (genes, or the EC/locus that links them) in one pathway
    x = cluster_genes.merge(gene_pw, on="locus").groupby(["Cluster", "plantcyc_id"])[count].nunique()
    return x[x >= 2].reset_index()["Cluster"].nunique()

members = clusters.assign(m=clusters["Members"].str.split()).explode("m")
members["locus"] = members["m"].str.extract(mm.LOCUS)[0]
cluster_genes = members.dropna(subset=["locus"])[["Cluster", "locus"]].drop_duplicates()
gene_pw = gmap[["locus", "plantcyc_id", "via"]].drop_duplicates()

rng = np.random.default_rng(0)
shuffles = [cluster_genes.assign(locus=rng.permutation(cluster_genes["locus"].values)) for _ in range(100)]
observed = coherent_clusters(cluster_genes, gene_pw)
null = [coherent_clusters(sh, gene_pw) for sh in shuffles]
# Stricter: the genes must reach the pathway through >= 2 DIFFERENT ECs/loci (excludes co-expressed paralogs of one enzyme)
observed_strict = coherent_clusters(cluster_genes, gene_pw, count="via")
null_strict = [coherent_clusters(sh, gene_pw, count="via") for sh in shuffles]
say(f"""**Findings** (same 100 seeded shuffles for both counts)
- **≥ 2 genes** of the same PlantMetWiki pathway: observed **{observed}** clusters; random mean **{np.mean(null):.1f}**
  (range {min(null)}–{max(null)}; {sum(n >= observed for n in null)} of 100 shuffles reach the observed value).
- **≥ 2 different enzymes** (distinct EC numbers or loci) of the same pathway: observed **{observed_strict}**; random mean **{np.mean(null_strict):.1f}**
  (range {min(null_strict)}–{max(null_strict)}; {sum(n >= observed_strict for n in null_strict)} of 100 reach it).""")

**Findings** (same 100 seeded shuffles for both counts)
- **≥ 2 genes** of the same PlantMetWiki pathway: observed **37** clusters; random mean **10.4**
  (range 4–17; 0 of 100 shuffles reach the observed value).
- **≥ 2 different enzymes** (distinct EC numbers or loci) of the same pathway: observed **26**; random mean **9.1**
  (range 5–15; 0 of 100 reach it).

In [13]:
co = (cluster_genes.merge(gmap[["locus", "plantcyc_id", "pathway", "tier", "via"]].drop_duplicates(), on="locus")
      .groupby(["Cluster", "plantcyc_id", "pathway"])
      .agg(n_genes=("locus", "nunique"), n_distinct_ec_or_loci=("via", "nunique"),
           genes=("locus", lambda s: ", ".join(sorted(set(s)))), via=("via", lambda s: ", ".join(sorted(set(s)))),
           tiers=("tier", lambda s: ", ".join(sorted(set(s))))).reset_index())
co = co[co["n_genes"] >= 2].merge(clusters[["Cluster", "Size", "P-value"]], on="Cluster")
co = co.merge(res["pathway_classes"][["category"]], left_on="plantcyc_id", right_index=True, how="left")
co.sort_values(["n_distinct_ec_or_loci", "n_genes", "P-value"], ascending=[False, False, True]).head(25)

,Cluster,plantcyc_id,pathway,n_genes,n_distinct_ec_or_loci,genes,via,tiers,Size,P-value,category
53,161,PWY-5464,"superpathway of cytosolic glycolysis (plants), pyruvate dehydrogen...",3,3,"Solyc02g084440, Solyc05g053300, Solyc07g005390","EC 1.2.1.9, EC 1.8.1.4, EC 4.1.2.13",gene_ec,78,0.965894,primary
21,67,PWY-5464,"superpathway of cytosolic glycolysis (plants), pyruvate dehydrogen...",3,3,"Solyc03g114500, Solyc09g075450, Solyc12g099100","EC 1.8.1.4, EC 4.2.1.11, EC 4.2.1.2",gene_ec,34,0.999460,primary
81,803,PWY18C3-1,"cholesterol biosynthesis (plants, early side-chain reductase)",2,3,"Solyc02g086180, Solyc09g009040","C14-R, C5-SD2, EC 1.3.1.70","gene_ec, gene_locus",5,0.894962,primary
61,171,PWY-7345,superpathway of anaerobic sucrose degradation,3,2,"Solyc09g009260, Solyc10g076510, Solyc10g083570","EC 4.1.1.1, EC 4.1.2.13",gene_ec,15,0.999475,primary
8,18,P165-PWY,superpathway of purines degradation in plants,2,2,"Solyc11g006550, Solyc12g018990","EC 1.7.3.3, EC 3.2.2.2",gene_ec,177,0.000000,primary
67,219,PWY-5084,2-oxoglutarate decarboxylation to succinyl-CoA,2,2,"Solyc05g054640, Solyc07g064800","EC 1.2.4.2, EC 2.3.1.61",gene_ec,71,0.000000,primary
66,210,PWY-7345,superpathway of anaerobic sucrose degradation,2,2,"Solyc09g009260, Solyc10g076510","EC 4.1.1.1, EC 4.1.2.13",gene_ec,39,0.011349,primary
0,1,GLUCONEO-PWY,gluconeogenesis I,2,2,"Solyc05g008600, Solyc10g005510","EC 1.2.1.12, EC 4.1.2.13",gene_ec,28,0.631538,primary
1,1,GLYCOLYSIS,glycolysis I (from glucose 6-phosphate),2,2,"Solyc05g008600, Solyc10g005510","EC 1.2.1.12, EC 4.1.2.13",gene_ec,28,0.631538,primary
2,1,PWY-1042,glycolysis IV,2,2,"Solyc05g008600, Solyc10g005510","EC 1.2.1.12, EC 4.1.2.13",gene_ec,28,0.631538,primary


**Interpretation**
- **Genes of one PlantMetWiki pathway cluster together more than expected by chance.** The MEANtools gene modules do capture pathway structure that PlantMetWiki knows about. Examples are central carbon metabolism, the mevalonate pathway and the GAME genes of α-tomatine biosynthesis.
- **Paralogs inflate the simple count.** Part of the signal comes from co-expressed paralogs with the same EC number (often tandem duplicates). The stricter count, which needs ≥ 2 *different* enzymes, is the better measure of pathway coherence.
- These pathway-coherent clusters are the best starting points for looking at their co-clustered metabolite features. Even without an identification, a feature in such a cluster is a candidate product or substrate of that pathway.

## 7. Case study: falcarindiol and crepenynate

Jeon et al. discovered a tomato gene cluster for **falcarindiol**, an acetylenic fatty-acid phytoalexin. What can we see of it?

In [14]:
pe = pmw_genes["pathway_ec"]
crep = pe[pe["pathway"].str.contains("crepenyn|falcarin", case=False)]
display(crep)
fal = pd.Series({"M283T2030 ([M+Na]+ of falcarindiol) in fungal correlations": "M283T2030" in set(res["corr"]["metabolite"]),
                 "M225T2033 ([M+H-2H2O]+ of falcarindiol) in fungal correlations": "M225T2033" in set(res["corr"]["metabolite"]),
                 "falcarindiol mass (260.178) matches a PlantMetWiki metabolite": bool(((mm.neutral_mass_from_inchi("InChI=1S/C17H24O2/x") -
                     pmw_genes["metabolite_inchi"]["inchi"].map(mm.neutral_mass_from_inchi)).abs() < 0.003).any()),
                 "tomato UniProt proteins with the crepenynate EC(s)": int(uniprot["ec"].isin(set(crep["ec"])).sum())},
               name="value").to_frame()
fal

,pc,ec,plantcyc_id,pathway
345,PC1085,1.14.19.39,PWY-6013,crepenynate biosynthesis


,value
M283T2030 ([M+Na]+ of falcarindiol) in fungal correlations,False
M225T2033 ([M+H-2H2O]+ of falcarindiol) in fungal correlations,False
falcarindiol mass (260.178) matches a PlantMetWiki metabolite,False
tomato UniProt proteins with the crepenynate EC(s),0


**Comments**
- **PlantMetWiki contains the first committed step.** *Crepenynate biosynthesis* (the FAD2-type acetylenase, EC 1.14.19.39) is in PlantMetWiki, but falcarindiol itself and the downstream steps are not.
- **The EC route can't reach the tomato acetylenases.** No tomato UniProt entry carries that EC, because the tomato genes are annotated as generic fatty-acid desaturases.
- **The metabolite side is missing too.** The falcarindiol features were not retained in the fungal MEANtools output.
- **Together this is exactly the situation where a MEANtools association is *new* knowledge.** The evidence model has to be able to express it without PlantMetWiki support. A curated PlantMetWiki pathway for falcarindiol (from Jeon et al.) would be a valuable addition.

## 8. Towards an evidence model for MEANtools gene–metabolite associations

Each gene–metabolite pair in a functional cluster becomes one **association** with its evidence attached.
The script writes these as RDF to [`results/associations.ttl`](results/associations.ttl): all pairs from significant clusters (p < 0.05), plus any pair with PlantMetWiki support.
All terms below were checked in the EBI Ontology Lookup Service.

| What | Term | Notes |
|---|---|---|
| the association | `SIO:001257` *chemical-gene association* | links the Solyc gene and the metabolite feature (`SIO:000628` *refers to*) |
| has evidence | `SIO:000772` *has evidence* | points to an evidence node |
| evidence type | `ECO:0007590` *high throughput expression pattern evidence used in automatic assertion* | closest existing ECO term (see gap below) |
| method | `EDAM:operation_3463` *Expression correlation analysis* + `EDAM:operation_3432` *Clustering* | the MEANtools run, with `SWO:0004000` *has version* "1.0.0" and `prov:used` the Zenodo record |
| correlation | `STATO:0000280` *Pearson's correlation coefficient* | MEANtools default method |
| significance | `OBI:0000175` *p-value* | of the correlation; the cluster p-value is recorded separately |
| mutual rank, edge weight, decay rate, cluster id | *no standard terms* | placeholder properties (`lodp:`) for now |
| PlantMetWiki support | `lodp:supportedByPlantMetWikiPathway` + gene mapping tier | optional; absent for new associations |

**Gap: ECO has no term for metabolite–transcript co-expression / correlation evidence.**
A term request to ECO ([github.com/evidenceontology/evidenceontology](https://github.com/evidenceontology/evidenceontology/issues)) could propose, for example:
*"metabolite–transcript correlation evidence"*, with a child *"… used in automatic assertion"*, under *expression pattern evidence*.

In [15]:
ttl = (mm.RESULTS / "associations.ttl").read_text()
n_assoc = ttl.count("a sio:SIO_001257")
say(f"`results/associations.ttl` holds **{n_assoc}** associations ({len(ttl) / 1e6:.1f} MB). The first one:")
first = ttl.index("lodp:assoc_")
print(ttl[:ttl.index('.\n', ttl.index('lodp:decayRate')) + 2])
print(ttl[first:ttl.index(' .\n', first) + 3])

`results/associations.ttl` holds **4068** associations (2.5 MB). The first one:

@prefix lodp: <https://example.org/lod-pathways/pilot/> .   # placeholder namespace
@prefix sio: <http://semanticscience.org/resource/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix edam: <http://edamontology.org/> .
@prefix swo: <http://www.ebi.ac.uk/swo/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

lodp:meantools_run_fungal_DR10 a edam:operation_3463 , edam:operation_3432 ;   # expression correlation analysis ; clustering
    rdfs:label "MEANtools functional clustering, fungal treatment, decay rate 10" ;
    prov:used <https://zenodo.org/records/15697913> ;
    lodp:software "MEANtools" ; swo:SWO_0004000 "1.0.0" ;   # has version
    lodp:decayRate 10 .

lodp:assoc_fungal_DR10_C9_Solyc03g123680_M772T844 a sio:SIO_001257 ;   # chemical-gene association
    sio:SIO_000628 <https://plants.ensembl.org/Solanum_lycopersicum/Gene/Summary?g=Solyc03g123680> , lodp:featur

## 9. Summary

In [16]:
say(f"""
- **Genes.** {gmap['locus'].nunique()} of {len(genes)} clustered genes ({100 * gmap['locus'].nunique() / len(genes):.0f}%) reach a PlantMetWiki pathway, almost all through EC numbers.
  Only {gmap.loc[gmap['tier'] == 'gene_locus', 'locus'].nunique()} tomato loci are in PlantMetWiki directly.
- **Metabolites.** {mmap['feature'].nunique()} of {len(feats)} clustered features ({100 * mmap['feature'].nunique() / len(feats):.0f}%) match a PlantMetWiki pathway metabolite by MS1 mass only.
  MEANtools' own mass-annotation table in the SQLite file is not mass-consistent and was not used.
- **Associations.** {obs} of {both.sum()} fully mapped cluster pairs share a PlantMetWiki pathway (≈{exp_frac * both.sum():.1f} expected by chance).
  PlantMetWiki therefore rarely confirms MEANtools associations in this dataset.
- **Gene modules.** {observed} clusters group ≥ 2 genes of the same PlantMetWiki pathway (random: {np.mean(null):.1f});
  {observed_strict} with ≥ 2 different enzymes (random: {np.mean(null_strict):.1f}). MEANtools gene modules do reflect known pathways.
- **Evidence model.** Associations are exported as `SIO:001257` with `ECO:0007590` evidence, Pearson r, p-value, mutual rank and cluster significance.
  ECO lacks a dedicated metabolite–transcript correlation term.
""")


- **Genes.** 311 of 3406 clustered genes (9%) reach a PlantMetWiki pathway, almost all through EC numbers.
  Only 14 tomato loci are in PlantMetWiki directly.
- **Metabolites.** 43 of 137 clustered features (31%) match a PlantMetWiki pathway metabolite by MS1 mass only.
  MEANtools' own mass-annotation table in the SQLite file is not mass-consistent and was not used.
- **Associations.** 0 of 721 fully mapped cluster pairs share a PlantMetWiki pathway (≈5.3 expected by chance).
  PlantMetWiki therefore rarely confirms MEANtools associations in this dataset.
- **Gene modules.** 37 clusters group ≥ 2 genes of the same PlantMetWiki pathway (random: 10.4);
  26 with ≥ 2 different enzymes (random: 9.1). MEANtools gene modules do reflect known pathways.
- **Evidence model.** Associations are exported as `SIO:001257` with `ECO:0007590` evidence, Pearson r, p-value, mutual rank and cluster significance.
  ECO lacks a dedicated metabolite–transcript correlation term.


**Next steps**
- [ ] Better metabolite identities: run SIRIUS or a spectral library search on the MTBLS1039 MS/MS data, then reuse the evidence tiers of the first pilot.
- [ ] Test the bacterial-treatment output (2.6 GB): `mm.build("bacterial", …)`. Check whether the falcarindiol features are retained there.
- [ ] Better gene function: add Pfam/InterPro-based EC predictions (MEANtools ships a Pfam–EC map) or orthology to Arabidopsis genes in PlantMetWiki.
- [ ] Reaction-level matching: gene EC and metabolite as substrate/product of the *same* reaction, not just the same pathway.
- [ ] Propose the ECO term; define `lodp:` properties (mutual rank, decay rate, cluster) in a small vocabulary.
- [ ] Report the `NPDB_annotations` / `mm_low` issue to the MEANtools authors.